# OLD

In [ ]:
import sys
from pathlib import Path

sys.path.append(f"{Path().absolute().parent}")

In [ ]:
import warnings

# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning, module="gpytorch")

In [ ]:
from apps.mobility_robustness_optimization.mobility_robustness_optimization import *
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO
from apps.mobility_robustness_optimization.mro_rl import ReinforcedMRO

In [ ]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 100,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 5,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 5,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [ ]:
topology = pd.read_csv("data/sim_data/topology.csv")
ue_data = pd.read_csv('data/sim_data/UE_Data_20UE_100ticks.csv')

topology.loc[topology["cell_id"] == "cell_1", "cell_lat"] = -90
topology.loc[topology["cell_id"] == "cell_2", "cell_lat"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lat"] = 90

topology.loc[topology["cell_id"] == "cell_1", "cell_lon"] = -180
topology.loc[topology["cell_id"] == "cell_2", "cell_lon"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lon"] = 180

topology.loc[topology["cell_id"] == "cell_1", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_2", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_3", "cell_carrier_freq_mhz"] = 2100

In [ ]:
ue_data.head()

In [ ]:
ue_data2 = pd.read_csv('data/sim_data/100UE_500Ticks.csv')
ue_data2.head()

In [ ]:
topology['cell_id'].unique()

## Simple MRO

In [ ]:
mro = SimpleMRO(params, topology)

In [ ]:
mro.bayesian_digital_twins

In [ ]:
mro.train_or_update_rf_twin(ue_data)

In [ ]:
mro.bayesian_digital_twins

In [ ]:
hyst,ttt = mro.solve()

In [ ]:
mro.bayesian_digital_twins

In [ ]:
# mro.train_or_update_rf_twin(ue_data2)
mro.train_or_update_rf_twin(ue_data) # this effectively updates the model with the new data as bdt exists in the model already

In [ ]:
hyst,ttt = mro.solve()

# ! ISSUE: when using train_or_update_rf_twin() to train bdt for the first time, solve() works fine. But when using it to update the model with new data, solve() fails with the error below:

In [ ]:
ue_data_synthetic_rxdbm = ue_data2 = pd.read_csv('data/sim_data/100UE_500Ticks.csv')
ue_data_synthetic_rxdbm

In [ ]:
random_numbers = 99 + (np.random.rand(len(ue_data_synthetic_rxdbm)) * 2)
synthetic_rxpowerdbm = [f"{val:.6f}" for val in random_numbers]

ue_data_synthetic_rxdbm['cell_rxpower_dbm'] = synthetic_rxpowerdbm

ue_data_synthetic_rxdbm

In [ ]:
mro.train_or_update_rf_twin(ue_data_synthetic_rxdbm)

In [ ]:
hyst,ttt = mro.solve()

## ue_data with rx_pwr col

In [ ]:
mro3 = SimpleMRO(params, topology)

In [ ]:
ue_data3 = ue_data.copy()

In [ ]:
random_numbers = 99 + (np.random.rand(len(ue_data3)) * 2)
synthetic_rxpowerdbm = [f"{val:.6f}" for val in random_numbers]

ue_data3['cell_rxpower_dbm'] = synthetic_rxpowerdbm

ue_data3.head()

In [ ]:
mro3.train_or_update_rf_twin(ue_data3)

In [ ]:
ue_data3.head()

In [ ]:
mro3.combined_df.head()

# ! ISSUE: need more low level instruction for the flow of rx_power_dbm inside the new_data otherwise, the cartesian product df makes no sense and the further codebase is built on top of that

In [ ]:
hyst,ttt = mro3.solve()

## RL MRO

In [ ]:
topology = pd.read_csv("data/sim_data/topology.csv")
ue_data = pd.read_csv('data/sim_data/UE_Data_20UE_100ticks.csv')

In [ ]:
mro = ReinforcedMRO(params, topology)

In [ ]:
mro.train_or_update_rf_twin(ue_data)

In [ ]:
mro.train_or_update_rf_twin(ue_data2)

In [ ]:
mro.train_or_update_rf_twin(ue_data_synthetic_rxdbm)

In [ ]:
hyst,ttt = mro.solve()

# Testing Scenarios

this nb is updated for final checking on all scenarios.

## Intitializing

In [ ]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

import pandas as pd
from notebooks.radp_library import (preprocess_ue_data)
from notebooks.radp_library import get_ue_data
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO
from apps.mobility_robustness_optimization.mro_rl import ReinforcedMRO

In [ ]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 100,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 5,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 5,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [ ]:
new_data = pd.read_csv('data/sim_data/UE_data_20UE_100ticks.csv')
topology = pd.read_csv('data/sim_data/topology.csv')
topology.loc[topology["cell_id"] == "cell_1", "cell_lat"] = -90
topology.loc[topology["cell_id"] == "cell_2", "cell_lat"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lat"] = 90

topology.loc[topology["cell_id"] == "cell_1", "cell_lon"] = -180
topology.loc[topology["cell_id"] == "cell_2", "cell_lon"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lon"] = 180

topology.loc[topology["cell_id"] == "cell_1", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_2", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_3", "cell_carrier_freq_mhz"] = 2100

new_data.drop(columns=['mock_ue_id', 'tick'], inplace=True)
new_data

In [ ]:
topology

## Scenario 1:
- `new_data` only has `[Latitude, Longitude]`, doesn't have any `rxpower` data
- `bdt` is empty.

### Simple MRO

In [ ]:
# calling the preprocess_ue_data function to get rx power data in cartesian``
input_data = preprocess_ue_data(new_data, topology)
input_data

In [ ]:
mro = SimpleMRO(params, topology)

In [ ]:
mro.bayesian_digital_twins

In [ ]:
# train bdt from scratch
mro.train_or_update_rf_twin(input_data)

In [ ]:
mro.bayesian_digital_twins

In [ ]:
mro.save_bdt()

In [ ]:
# adjust n_epochs for better performance
mro.solve(n_epochs=3)

In [ ]:
# loading new UE data assuming user have new observations now.
new_ue = get_ue_data(params)
new_ue.rename(columns={"lon": "longitude", "lat": "latitude"}, inplace=True)

# calling the preprocess_ue_data function to get rx power data in cartesian
input_data = preprocess_ue_data(new_ue, topology)
input_data

In [ ]:
# update bdt with new data
mro.train_or_update_rf_twin(input_data)

In [ ]:
# adjust n_epochs for better performance
mro.solve(n_epochs=3)

### RL MRO

In [ ]:
input_data = preprocess_ue_data(new_data, topology)
input_data

In [ ]:
rl_mro = ReinforcedMRO(params, topology)

In [ ]:
rl_mro.bayesian_digital_twins

In [ ]:
# train bdt from scratch
rl_mro.train_or_update_rf_twin(input_data)

In [ ]:
rl_mro.bayesian_digital_twins

In [ ]:
rl_mro.save_bdt()

In [ ]:
# adjust total_timesteps for better performance
rl_mro.solve(total_timesteps=2)

In [ ]:
# loading new UE data assuming user have new observations now.
new_ue = get_ue_data(params)
new_ue.rename(columns={"lon": "longitude", "lat": "latitude"}, inplace=True)

# calling the preprocess_ue_data function to get rx power data in cartesian
input_data = preprocess_ue_data(new_ue, topology)
input_data

In [ ]:
rl_mro.train_or_update_rf_twin(input_data)

In [ ]:
# adjust total_timesteps for better performance
rl_mro.solve(total_timesteps=2)

## Scenario 2:
- `new_data` has `[Latitude, Longitude, cell_id, cell_rxpower_dbm]`
- `bdt` is empty.

### Simple MRO

In [ ]:
# create data with rx power
input_data = preprocess_ue_data(new_data, topology)
new_data_with_rxpower = input_data.drop(columns=['cell_lat', 'cell_lon', 'cell_az_deg', "cell_carrier_freq_mhz", 'log_distance'])
new_data_with_rxpower

In [ ]:
mro = SimpleMRO(params, topology)

In [ ]:
mro.bayesian_digital_twins

In [ ]:
# train bdt from scratch
mro.train_or_update_rf_twin(new_data_with_rxpower)

In [ ]:
mro.bayesian_digital_twins

In [ ]:
mro.save_bdt()

In [ ]:
# adjust n_epochs for better performance
mro.solve(n_epochs=3)

In [ ]:
# loading new UE data assuming user have new observations now.
new_ue = get_ue_data(params)
new_ue.rename(columns={"lon": "longitude", "lat": "latitude"}, inplace=True)

# create data with rx power
new_ue_with_rxpwr = preprocess_ue_data(new_ue, topology)
new_ue_with_rxpwr = new_ue_with_rxpwr.drop(columns=['cell_lat', 'cell_lon', 'cell_az_deg', "cell_carrier_freq_mhz", 'log_distance', 'tick', 'mock_ue_id'])

# calling the preprocess_ue_data function to get rx power data in cartesian
input_data = new_ue_with_rxpwr.copy()
input_data

In [ ]:
# update bdt with new data
mro.train_or_update_rf_twin(input_data)

In [ ]:
# adjust n_epochs for better performance
mro.solve(n_epochs=3)

### RL MRO

In [ ]:
# create data with rx power
input_data = preprocess_ue_data(new_data, topology)
new_data_with_rxpower = input_data.drop(columns=['cell_lat', 'cell_lon', 'cell_az_deg', "cell_carrier_freq_mhz", 'log_distance'])
new_data_with_rxpower

In [ ]:
rl_mro = ReinforcedMRO(params, topology)

In [ ]:
rl_mro.bayesian_digital_twins

In [ ]:
# update bdt with new data
rl_mro.train_or_update_rf_twin(new_data_with_rxpower)

In [ ]:
rl_mro.bayesian_digital_twins

In [ ]:
rl_mro.save_bdt()

In [ ]:
# adjust total_timesteps for better performance
rl_mro.solve(total_timesteps=2)

In [ ]:
# loading new UE data assuming user have new observations now.
new_ue = get_ue_data(params)
new_ue.rename(columns={"lon": "longitude", "lat": "latitude"}, inplace=True)

# create data with rx power
new_ue_with_rxpwr = preprocess_ue_data(new_ue, topology)
new_ue_with_rxpwr = new_ue_with_rxpwr.drop(columns=['cell_lat', 'cell_lon', 'cell_az_deg', "cell_carrier_freq_mhz", 'log_distance', 'tick', 'mock_ue_id'])

# calling the preprocess_ue_data function to get rx power data in cartesian
input_data = new_ue_with_rxpwr.copy()
input_data

In [ ]:
# update bdt with new data
rl_mro.train_or_update_rf_twin(input_data)

In [ ]:
rl_mro.solve(total_timesteps=2)

## Scenario 3:
- `new_data` only has `[Latitude, Longitude]`, doesn't have any `rxpower` data
- `bdt` is populated.

### Simple MRO

In [ ]:
# calling the preprocess_ue_data function to get rx power data in cartesian``
input_data = preprocess_ue_data(new_data, topology)
input_data

In [ ]:
mro = SimpleMRO(params, topology,) # Either BDT paramter is passed or mro intiated and then mro.load

In [ ]:
mro.bayesian_digital_twins

In [ ]:
mro.load_bdt() # Load existing bdt

In [ ]:
mro.bayesian_digital_twins

In [ ]:
# dummy solve call to avoid fantasy error
mro.solve(n_epochs=3)

In [ ]:
# update bdt with new data
mro.train_or_update_rf_twin(input_data)

In [ ]:
# adjust n_epochs for better performance
mro.solve(n_epochs=3)

### RL MRO

In [ ]:
# calling the preprocess_ue_data function to get rx power data in cartesian``
input_data = preprocess_ue_data(new_data, topology)
input_data

In [ ]:
rl_mro = ReinforcedMRO(params, topology) # Either BDT paramter is passed or mro intiated and then mro.load

In [ ]:
rl_mro.bayesian_digital_twins

In [ ]:
rl_mro.load_bdt()

In [ ]:
rl_mro.bayesian_digital_twins

In [ ]:
# dummy solve call to avoid fantasy error
rl_mro.solve(total_timesteps=2)

In [ ]:
# update bdt with new data
rl_mro.train_or_update_rf_twin(input_data)

In [ ]:
# adjust n_epochs for better performance
rl_mro.solve(total_timesteps=2)

## Scenario 4:
- `new_data` has `[Latitude, Longitude, cell_id, cell_rxpower_dbm]`
- `bdt` is populated.

### Simple MRO

In [ ]:
# create data with rx power
input_data = preprocess_ue_data(new_data, topology)
new_data_with_rxpower = input_data.drop(columns=['cell_lat', 'cell_lon', 'cell_az_deg', "cell_carrier_freq_mhz", 'log_distance'])
new_data_with_rxpower

In [ ]:
input_data = new_ue_with_rxpwr.copy()
input_data

In [ ]:
mro = SimpleMRO(params, topology,) # Either BDT paramter is passed or mro intiated and then mro.load

In [ ]:
mro.bayesian_digital_twins

In [ ]:
mro.load_bdt() # Load existing bdt

In [ ]:
mro.bayesian_digital_twins

In [ ]:
# dummy solve call to avoid fantasy error
mro.solve(n_epochs=3)

In [ ]:
# update bdt with new data
mro.train_or_update_rf_twin(input_data)

In [ ]:
# adjust n_epochs for better performance
mro.solve(n_epochs=3)

### RL MRO

In [ ]:
# create data with rx power
input_data = preprocess_ue_data(new_data, topology)
new_data_with_rxpower = input_data.drop(columns=['cell_lat', 'cell_lon', 'cell_az_deg', "cell_carrier_freq_mhz", 'log_distance'])
new_data_with_rxpower

In [ ]:
input_data = new_ue_with_rxpwr.copy()
input_data

In [ ]:
rl_mro = ReinforcedMRO(params, topology) # Either BDT paramter is passed or mro intiated and then mro.load

In [ ]:
rl_mro.bayesian_digital_twins

In [ ]:
rl_mro.load_bdt() # Load existing bdt

In [ ]:
rl_mro.bayesian_digital_twins

In [ ]:
# dummy solve call to avoid fantasy error
rl_mro.solve(total_timesteps=2)

In [ ]:
# update bdt with new data
rl_mro.train_or_update_rf_twin(input_data)

In [ ]:
# adjust total_timesteps for better performance
rl_mro.solve(total_timesteps=2)